# AutoNavLog

**地上準備専用**。このNotebookの値は運航資料や完成帳票ではありません。警告と根拠を確認し、別添8-1へ手書きで清書してください。

1 Projectを複数Notebookから同時編集しないでください。

## 1. 開始・環境確認
## 2. Projectの作成・読込
## 3. コース取込・編集
## 4. 飛行計画入力
## 5. Forecast Run選択
## 6. 計算実行
## 7. 警告・未確定項目の解消
## 8. 清書ビュー
## 9. 保存・Snapshot作成

In [ ]:
from __future__ import annotations

import ctypes.util
import hashlib
import importlib.util
import json
import subprocess
import sys
from pathlib import Path

VERSION = "0.1.0"
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    RELEASES_ROOT = Path("/content/drive/MyDrive/AutoNavLog/releases")
    available = sorted(
        (
            path.name
            for path in RELEASES_ROOT.iterdir()
            if path.is_dir() and all(part.isdigit() for part in path.name.split("."))
        ),
        key=lambda value: tuple(int(part) for part in value.split(".")),
    )
    if available and available[-1] != VERSION:
        print(f"新版 {available[-1]} があります。このNotebookは {VERSION} を継続使用します。")
    RELEASE_ROOT = RELEASES_ROOT / VERSION
    manifest = json.loads((RELEASE_ROOT / "release-manifest.json").read_text())
    for item in manifest["files"]:
        path = RELEASE_ROOT / item["path"]
        if hashlib.sha256(path.read_bytes()).hexdigest() != item["sha256"]:
            raise RuntimeError(f"配布ファイルのSHA-256が一致しません: {item['path']}")
    if ctypes.util.find_library("eccodes") is None:
        subprocess.run(["apt-get", "update", "-qq"], check=True)
        subprocess.run(["apt-get", "install", "-y", "-qq", "libeccodes0"], check=True)
    wheels = sorted(str(path) for path in (RELEASE_ROOT / "wheels").glob("*.whl"))
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *wheels], check=True)
else:
    RELEASE_ROOT = Path.cwd()
print(f"AutoNavLog {VERSION} / release root: {RELEASE_ROOT}")

In [ ]:
from autonavlog.application import CalculationService, ProjectService
from autonavlog.performance import PerformanceRepository
from autonavlog.presentation import AutoNavLogApp
from autonavlog.storage import AirportRepository, LocalProjectRepository
from autonavlog.storage.drive import GoogleDriveProjectRepository
from autonavlog.weather import FakeWeatherProvider
from autonavlog.weather.msm_adapter import MsmWeatherProvider

if IN_COLAB:
    app_data = RELEASE_ROOT / "data" / "autonavlog"
    repository = GoogleDriveProjectRepository("/content/drive/MyDrive")
    weather = MsmWeatherProvider(
        "/content/drive/MyDrive/AutoNavLog/cache",
        RELEASE_ROOT / "data" / "msm" / "terrain.npz",
    )
else:
    app_data = RELEASE_ROOT / "data"
    repository = LocalProjectRepository(RELEASE_ROOT / ".notebook-data" / "AutoNavLog")
    weather = FakeWeatherProvider()

airports = AirportRepository.from_csv(app_data / "airports" / "airports.csv")
performance = PerformanceRepository.from_directory(app_data / "performance")
calculation = CalculationService(airports, performance)
projects = ProjectService(repository)
app = AutoNavLogApp(projects, calculation, weather)
app.render()